# 5. 1-stage対凍結2-stage

Spatial S34・DRO/globalを固定し、1-stage → ERMからの凍結2-stageを比較する。

2-stageは同seedのM01親。30対30+30epochで、初期値・凍結・学習量を含む手順全体の差。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import rootutils
from IPython.display import display

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET, study_dir  # noqa: E402
from analysis.archive.race_known_architecture_objective_comparison.comparison import (  # noqa: E402
    METRICS,
    evaluate,
    group_ids,
    paired_deltas,
    plot_performance,
    plot_q,
    prediction_command,
    read_predictions,
    read_q_history,
    select_runs,
)

plt.style.use(STYLE_SHEET)
PACKAGE = study_dir("race_known_architecture_objective_comparison")
TOPIC = "05_one_stage_vs_frozen_two_stage"
SEEDS = [42, 43, 44]
CONDITIONS = {
    "M04": "spatial-lora-chexpert-age-race-group-dro-race-known",
    "M06": "spatial-lora-chexpert-from-resnet-age-race-group-dro-race-known",
}
LABELS = {"M04": "1-stage", "M06": "Frozen 2-stage"}
PAIRS = [["M04", "M06"]]
RESULTS = PACKAGE / "results" / TOPIC
FIGURES = PACKAGE / "figures" / TOPIC
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## 入力

成功run・best checkpointと学習時manifestのtrain/val CSVをSHA256で照合する。testは共通CSVの現在のSHA256を記録し、cacheの行順・targetを検証する。cache欠損時は提示コマンドをrepo rootで実行（GPU自動起動なし）。

In [ ]:
provenance, frame, directories = select_runs(PACKAGE, CONDITIONS, SEEDS)
provenance.to_csv(RESULTS / "provenance.csv", index=False)
display(
    provenance[
        [
            "condition",
            "seed",
            "run_id",
            "checkpoint",
            "objective",
            "class_weight_scope",
            "freeze_backbone",
            "parent_run",
        ]
    ]
)
command = prediction_command(PACKAGE, directories, device="cuda:0")
print(command or "全予測cacheが有効")

## 全体性能・公平性

AUROC・worst-group AUROCが主指標。BACCとgapは補助。固定argmaxでEopp0=TNR gap、Eopp1=TPR gap、Eodds=(TPR gap+FPR gap)/2。平均±標準偏差は3seed間のばらつきで、有意差を意味しない。

In [ ]:
caches = read_predictions(PACKAGE, directories, frame)
summary, subgroup = evaluate(caches, frame)
summary.to_csv(RESULTS / "metrics_by_seed.csv", index=False)
subgroup.to_csv(RESULTS / "subgroup_by_seed.csv", index=False)
condition_stats = summary.groupby("condition", sort=False)[METRICS].agg(["mean", "std"])
condition_stats.to_csv(RESULTS / "condition_mean_std.csv")
display(condition_stats.round(4))

## 同seedの差

比較条件−baselineをseed別に示す。AUROC/BACCは正、gap/Eopp/Eoddsは負が改善方向。gap縮小はworst改善かbest低下かを併せて読む。

In [ ]:
delta = paired_deltas(summary, PAIRS)
delta.to_csv(RESULTS / "paired_deltas.csv", index=False)
delta_stats = delta.groupby("pair", sort=False)[METRICS].agg(["mean", "std"])
delta_stats.to_csv(RESULTS / "paired_delta_mean_std.csv")
display(delta.round(4))
display(delta_stats.round(4))

## 8群の性能

年齢<65／≥65 × White・Asian・Black・Others。件数・陽性/陰性数を添える。欠損属性は群評価のみ除外し、single-class群の指標はNaNのまま扱う。

In [ ]:
counts = subgroup[subgroup.condition == next(iter(CONDITIONS))].query("seed == 42")[
    ["group_id", "group", "n", "n_positive", "n_negative"]
]
counts.to_csv(RESULTS / "group_counts.csv", index=False)
print(f"test全体: {len(frame)}, 8群対象: {(group_ids(frame) >= 0).sum()}, 属性欠損除外: {(group_ids(frame) < 0).sum()}")
display(counts.set_index("group"))
group_stats = subgroup.groupby(["group", "condition"], sort=False)[["auroc", "bacc"]].agg(["mean", "std"])
group_stats.to_csv(RESULTS / "subgroup_mean_std.csv")
display(group_stats.round(4))
fig = plot_performance(summary, subgroup, LABELS)
fig.savefig(FIGURES / "performance.png", dpi=160)
plt.show()

## qのepoch推移

実線はepoch末q、破線はbest checkpoint epoch（0始まり）。q_mean・群内weighted CE・countもCSVへ保存する。qが大きい群とtestで悪い群が一致するとは限らない。

In [ ]:
q_history = read_q_history(directories, provenance)
q_history.to_csv(RESULTS / "q_history.csv", index=False)
fig = plot_q(q_history, LABELS)
fig.savefig(FIGURES / "q_end_epochs.png", dpi=160)
plt.show()
q_best = q_history[q_history.epoch == q_history.best_epoch].copy()
q_best.to_csv(RESULTS / "q_at_best.csv", index=False)
q_max = q_best.loc[q_best.groupby(["condition", "seed"]).q_end.idxmax(), ["condition", "seed", "group"]].rename(
    columns={"group": "max_q_group"}
)
loss_max = q_best.loc[q_best.groupby(["condition", "seed"]).loss.idxmax(), ["condition", "seed", "group"]].rename(
    columns={"group": "max_train_loss_group"}
)
dro_groups = subgroup[subgroup.condition.isin(q_best.condition.unique())]
test_worst = dro_groups.loc[
    dro_groups.groupby(["condition", "seed"]).auroc.idxmin(), ["condition", "seed", "group"]
].rename(columns={"group": "worst_test_auroc_group"})
q_diagnostics = q_max.merge(loss_max, on=["condition", "seed"]).merge(test_worst, on=["condition", "seed"])
q_diagnostics.to_csv(RESULTS / "q_diagnostics.csv", index=False)
display(q_diagnostics)

## TODO

- M08→M10は[6冊目](06_resnet_baselines_vs_groupdro_architectures.ipynb)で比較。他条件：M12→M13、M17→M19、M18→M20、M14→M15。親M01からの変化は補助比較。
- 単独属性・患者単位の不確実性とq／群lossの詳細な対応を確認。3seedで方式の優劣を断定しない。